In [23]:
import pandas as pd
import yfinance as yf
import datetime
from datetime import date, timedelta
today = date.today()

d1 = today.strftime("%Y-%m-%d")
end_date = d1
d2 = date.today() - timedelta(days=5000)
d2 = d2.strftime("%Y-%m-%d")
start_date = d2

data = yf.download('NFLX', 
                      start=start_date, 
                      end=end_date, 
                      progress=False)
data["Date"] = data.index
#data.head()
data = data[["Date", "Open","High","Low","Close", "Volume"]]
data.reset_index(drop=True, inplace =True)
print(data.tail())
data = data.dropna()

C:\Users\user\AppData\Local\Temp\ipykernel_19144\609842916.py:13: FutureWarning:

YF.download() has changed argument auto_adjust default to True



Price        Date         Open         High          Low        Close  \
Ticker                    NFLX         NFLX         NFLX         NFLX   
3434   2025-07-14  1244.910034  1270.489990  1240.000000  1261.949951   
3435   2025-07-15  1262.000000  1271.219971  1243.239990  1260.270020   
3436   2025-07-16  1261.709961  1271.000000  1249.819946  1250.310059   
3437   2025-07-17  1253.000000  1277.500000  1244.800049  1274.170044   
3438   2025-07-18  1241.959961  1246.500000  1201.010010  1209.239990   

Price     Volume  
Ticker      NFLX  
3434     2781000  
3435     2801700  
3436     3227000  
3437     6469900  
3438    10682600  


In [24]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data["Date"],
                                        open=data["Open"], 
                                        high=data["High"],
                                        low=data["Low"], 
                                        close=data["Close"])])
figure.update_layout(title = "Netflix Stock Price Analysis", xaxis_rangeslider_visible =False)
    
              

In [25]:
correlation = data.corr()
corr_close_nflx = correlation[('Close', 'NFLX')]
print(corr_close_nflx.sort_values(ascending=False))

Price   Ticker
Close   NFLX      1.000000
Low     NFLX      0.999831
High    NFLX      0.999826
Open    NFLX      0.999600
Date              0.859041
Volume  NFLX     -0.464354
Name: (Close, NFLX), dtype: float64


In [26]:
x = data[["Open", "High", "Low", "Volume"]]
y = data["Close"]
x = x.to_numpy()
y = y.to_numpy()
y = y.reshape(-1, 1)

from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(x, y, 
                                                test_size=0.2, random_state=42)

In [28]:
from keras.models import Sequential
from keras.layers import Dense, LSTM
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape= (xtrain.shape[1], 1)))
model.add(LSTM(64, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

ModuleNotFoundError: No module named 'tensorflow'

In [26]:
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(xtrain, ytrain, batch_size=1, epochs=30)

Epoch 1/30
2758/2758 [==============================] - 25s 7ms/step - loss: 6564.7012
Epoch 2/30
2758/2758 [==============================] - 18s 6ms/step - loss: 952.3418
Epoch 3/30
2758/2758 [==============================] - 30s 11ms/step - loss: 405.8149
Epoch 4/30
2758/2758 [==============================] - 27s 10ms/step - loss: 559.6149
Epoch 5/30
2758/2758 [==============================] - 25s 9ms/step - loss: 365.3712
Epoch 6/30
2758/2758 [==============================] - 18s 6ms/step - loss: 295.4415
Epoch 7/30
2758/2758 [==============================] - 19s 7ms/step - loss: 280.0413
Epoch 8/30
2758/2758 [==============================] - 17s 6ms/step - loss: 173.0263
Epoch 9/30
2758/2758 [==============================] - 16s 6ms/step - loss: 281.0424
Epoch 10/30
2758/2758 [==============================] - 17s 6ms/step - loss: 194.6144
Epoch 11/30
2758/2758 [==============================] - 16s 6ms/step - loss: 155.4126
Epoch 12/30
2758/2758 [==========================

In [27]:
import numpy as np
#features = [Open, High, Low, Adj Close, Volume]
features = np.array([[401.970001, 427.700012, 398.200012, 20047500]])
model.predict(features)

1/1 [==============================] - 1s 1s/step


array([[408.56052]], dtype=float32)